# TEP dengan DAIL-SQL sebagai B — GraphRAG 1-Stage dan 2-Stage, per model

Notebook ini menghitung **TEP (Token Elasticity of Performance)** dari hasil run yang sudah disimpan, jadi **tidak butuh GPU** dan tidak menjalankan ulang pipeline. Logikanya ada di `src/experiments/tep_steps.py`.

$$\mathrm{TEP}_c = \frac{\Delta EX_c / EX_B}{\Delta T_c / T_B}, \qquad \Delta EX_c = EX_c - EX_B, \quad \Delta T_c = T_c - T_B$$

- **B = DAIL-SQL**, baseline skripsi (poin 38, 43). Sebelumnya B = GraphRAG 1-Stage; TEP itu sudah dihapus dari `eval_pipeline.ipynb` (poin 45).
- **c = GraphRAG 1-Stage** dan **c = GraphRAG 2-Stage**, masing-masing terhadap DAIL-SQL, untuk tiap model (3B, 7B, 14B).
- **Beberapa perbandingan k few-shot sekaligus** (poin 48), ditulis peneliti di `COMPARISONS` sebagai pasangan (k GraphRAG, k DAIL-SQL). Default: `GraphRAG K0 vs DAIL-SQL K9`, `GraphRAG K0 vs DAIL-SQL K0`, `GraphRAG K9 vs DAIL-SQL K9`.
- T = T_in + α × T_out per pertanyaan, lalu dirata-rata. Rumus dan fungsi sama dengan `src/metrics/tep.py` (`compute_tep()`), tidak diubah.

## File yang dibutuhkan

Yang dipakai sekarang adalah Excel di `outputs/tables/`, satu folder per metode dan k:

```
outputs/tables/
├── result_ourmethod/k0/  hasil_skripsi_3B_k0.xlsx, hasil_skripsi_7B_k0.xlsx, hasil_skripsi_14B_k0.xlsx
├── result_ourmethod/k9/  hasil_skripsi_3B_K9.xlsx, ...
├── result_dail/k0/       hasil_dail_3B_k0.xlsx, ...
└── result_dail/k9/       hasil_dail_3B_k9.xlsx, ...
```

Folder k baru (mis. `result_dail/k5/`) ikut terbaca tanpa mengubah kode; tambahkan pasangannya ke `COMPARISONS`. Zip hasil (kalau ada) diekstrak ke `outputs/models/`:

| Kondisi | Dari notebook | Taruh di |
|---|---|---|
| 1-Stage dan 2-Stage | `eval_pipeline.ipynb` → `hasil_<MODEL>.zip` | ekstrak ke `outputs/models/` (isinya folder `Qwen2.5-Coder-<MODEL>-Instruct/`) |
| DAIL-SQL | `eval_dail.ipynb` → `hasil_dail_<MODEL>.zip` | ekstrak ke `outputs/models/Qwen2.5-Coder-<MODEL>-Instruct/` (isinya `tables/dail/`, `raw_logs/`) |
| DAIL-SQL k lain | `eval_dail.ipynb` dengan `DAIL_K_SHOT` ≠ 9 → `hasil_dail_<MODEL>_k<k>.zip` | ekstrak ke subfolder sendiri, mis. `outputs/models/Qwen2.5-Coder-<MODEL>-Instruct/dail_k0/` |
| 1-Stage dan 2-Stage k lain | `eval_pipeline.ipynb` dengan `FEW_SHOT_K` lain → `hasil_<MODEL>.zip` | ekstrak ke folder sendiri, mis. `outputs/models/k9/` |
| 1-Stage dan 2-Stage (Excel) | `eval_pipeline.ipynb` → `hasil_skripsi_<MODEL>_k<k>.xlsx` | `outputs/tables/result_ourmethod/k<k>/` |
| DAIL-SQL (Excel) | `eval_dail.ipynb` → `hasil_dail_<MODEL>_k<k>.xlsx` | `outputs/tables/result_dail/k<k>/` |

Zip untuk k yang berbeda isinya bernama sama (`raw_logs/dail_log.json`, folder `Qwen2.5-Coder-<MODEL>-Instruct/`), jadi **jangan diekstrak ke folder yang sama**; k dibaca dari metadata, nama foldernya bebas.

Zip lebih baik daripada Excel: metadata run (nama model lengkap, `SAMPLE_RATIO`, k few-shot) ikut terbaca dan bisa dicek. Excel hanya dipakai kalau zip untuk model, kondisi, dan k itu tidak ada; angka EX, T_in, T_out-nya sama (sheet `Per Query`).

**k few-shot dari nama Excel:** Excel tidak menyimpan k, jadi k dibaca dari akhiran nama file `_K<k>` (huruf besar/kecil sama saja), mis. `hasil_skripsi_7B_K9.xlsx` atau `hasil_dail_7B_k0.xlsx`. Nama folder (`k0/`, `k9/`) tidak dibaca, hanya nama file. File tanpa akhiran memakai `EXCEL_DEFAULT_K` di sel konfigurasi (GraphRAG 0, DAIL-SQL 9). Dua file dengan model, kondisi, dan k yang sama = run itu dilewati dengan catatan.

## Pengecekan sebelum TEP dihitung

Baris yang tidak lolos tetap tampil dengan status `skipped`, tidak membuat notebook berhenti. Alasannya tidak ditampilkan di tabel, tapi tersimpan di `outputs/tables/hasil_tep.json` (field `Note` tiap baris).

- **Model sama** (ukuran model, dan nama lengkap kalau metadatanya ada).
- **Pertanyaan sama persis** (`query_id`, database, gold SQL). Artinya `SAMPLE_RATIO` dan seed harus sama di `eval_pipeline` dan `eval_dail`.
- **k sesuai pasangan di `COMPARISONS`.** Run dengan k yang tidak diminta tidak dipakai (dicatat di bagian 3).

## 1. Setup

In [1]:
# os untuk pindah folder
import os
# Path untuk menulis lokasi file
from pathlib import Path

# Path di notebook ini relatif terhadap root repo (outputs/models/, outputs/tables/).
# Dijalankan dari notebooks/: naik satu folder ke root repo
if Path.cwd().name == "notebooks":
    os.chdir("..")
# Lokasi root repo
REPO_ROOT = Path.cwd()
print("Repo root:", REPO_ROOT)

# --- Hanya untuk Kaggle (clone repo dari GitHub); dimatikan karena notebook ini dijalankan lokal ---
# import subprocess
# REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
# REPO_BRANCH = "main"
# def is_repo_root(p: Path) -> bool:
#     return (p / "src" / "experiments" / "tep_steps.py").is_file()
# if not is_repo_root(Path.cwd()):
#     if is_repo_root(Path.cwd().parent):
#         os.chdir("..")
#     else:
#         if not is_repo_root(Path("thesis-2803").resolve()):
#             subprocess.run(f"git clone -b {REPO_BRANCH} {REPO_URL}", shell=True, check=True)
#         os.chdir("thesis-2803")
# REPO_ROOT = Path.cwd()

Repo root: c:\William\Thesis\Code\thesis-2803


## 2. Import dan Konfigurasi

In [2]:
# sys untuk mendaftarkan lokasi kode
import sys
# Daftarkan root repo supaya modul src bisa di-import
sys.path.insert(0, str(REPO_ROOT))

# Konfigurasi utama: α dan k DAIL bawaan diambil dari sini, tidak ditulis ulang
from src.core.config import PipelineConfig
# Fungsi untuk menampilkan tabel di notebook
from src.experiments.result_tables import show
# Semua langkah notebook ini
from src.experiments import tep_steps as ts

# Konfigurasi bawaan
cfg = PipelineConfig()
# Bobot token output, sama untuk semua kondisi: T = T_in + α × T_out
ALPHA = cfg.token_output_weight
# Perbandingan yang dihitung: (k few-shot GraphRAG, k DAIL-SQL); urutan = urutan di tabel
COMPARISONS = [(0, 9), (0, 0), (9, 9)]
# k untuk Excel tanpa akhiran _K<k> di nama file: hasil_skripsi_<MODEL>.xlsx = GraphRAG k 0,
# hasil_dail_<MODEL>.xlsx = DAIL-SQL k bawaan (cfg.dail_k_shot = 9)
EXCEL_DEFAULT_K = {"pipeline": 0, "dail": cfg.dail_k_shot}
# Model yang diharapkan: 3B, 7B, 14B (yang belum ada hasilnya tetap muncul sebagai baris skipped)
MODELS = [ts.model_size(m) for m in cfg.llm_model_frame]
# Folder yang dicari (termasuk subfolder k0/, k9/, ...): zip yang diekstrak, Excel GraphRAG, Excel DAIL-SQL
INPUT_DIRS = [Path("outputs/models"), Path("outputs/tables/result_ourmethod"), Path("outputs/tables/result_dail")]
# Folder hasil notebook ini
OUT_DIR = Path("outputs/tables")

print(f"α = {ALPHA}, model = {MODELS}")
# Tampilkan nama tiap perbandingan
print("Perbandingan:", [ts.comparison_label(kg, kd) for kg, kd in COMPARISONS])

α = 3.0, model = ['3B', '7B', '14B']
Perbandingan: ['GraphRAG K0 vs DAIL-SQL K9', 'GraphRAG K0 vs DAIL-SQL K0', 'GraphRAG K9 vs DAIL-SQL K9']


## 3. Cari Hasil Run

Tabel di bawah menampilkan sumber yang dipakai per model, kondisi, dan k. Periksa kolom `Queries`, `Sample ratio`, `Few-shot k`, dan `k from` (`metadata`, `nama file`, atau `default`) sebelum membaca TEP.

In [3]:
# Cari hasil 1-Stage, 2-Stage, dan DAIL-SQL tiap model
runs, notes = ts.find_runs(INPUT_DIRS, EXCEL_DEFAULT_K)
# Tambahkan catatan run yang tidak dipakai perbandingan mana pun
notes += ts.unused_runs(runs, COMPARISONS)
# Tampilkan catatan (folder tidak ada, sumber ganda, k tidak diketahui, run yang tidak dipakai)
for note in notes:
    print("[INFO]", note)
# Tampilkan sumber yang dipakai
show(ts.sources_table(runs))

[INFO] Folder tidak ada: outputs\models


**Sources**

,Model,Condition,Few-shot k,k from,Source,Kind,Queries,Full model name,Sample ratio,Commit,Problem
0,3B,1-stage,0,nama file,outputs\tables\result_ourmethod\k0\hasil_skrip...,excel,1034,–,–,–,
1,3B,1-stage,9,nama file,outputs\tables\result_ourmethod\k9\hasil_skrip...,excel,1034,–,–,–,
2,3B,DAIL-SQL,0,nama file,outputs\tables\result_dail\k0\hasil_dail_3B_k0...,excel,1034,–,–,–,
3,3B,DAIL-SQL,9,nama file,outputs\tables\result_dail\k9\hasil_dail_3B_k9...,excel,1034,–,–,–,
4,3B,2-stage,0,nama file,outputs\tables\result_ourmethod\k0\hasil_skrip...,excel,1034,–,–,–,
5,3B,2-stage,9,nama file,outputs\tables\result_ourmethod\k9\hasil_skrip...,excel,1034,–,–,–,
6,7B,1-stage,0,nama file,outputs\tables\result_ourmethod\k0\hasil_skrip...,excel,1034,–,–,–,
7,7B,1-stage,9,nama file,outputs\tables\result_ourmethod\k9\hasil_skrip...,excel,1034,–,–,–,
8,7B,DAIL-SQL,0,nama file,outputs\tables\result_dail\k0\hasil_dail_7B_k0...,excel,1034,–,–,–,
9,7B,DAIL-SQL,9,nama file,outputs\tables\result_dail\k9\hasil_dail_7B_k9...,excel,1034,–,–,–,


## 4. TEP terhadap DAIL-SQL

Satu baris per model × perbandingan × pipeline. Urutan kolom mengikuti guide Dimensi 1: angka mentah dulu (EX dan T kedua kondisi), baru TEP. ΔEX dalam poin persen.

Pada `GraphRAG K0 vs DAIL-SQL K9`, ΔT memuat selisih token contoh few-shot (DAIL 9 contoh per pass, 2 pass), bukan hanya selisih schema. `K0 vs K0` dan `K9 vs K9` menyamakan jumlah contoh, jadi selisih yang tersisa berasal dari schema dan cara kerja masing-masing metode.

| Nilai TEP | Interpretasi (guide Bagian 3, Dimensi 1) |
|---|---|
| TEP < 0 | Skenario ideal: GraphRAG meningkatkan EX **dan** menurunkan Token Consumption dibanding baseline |
| TEP ≈ 0 (±0.05) | Efisiensi token tercapai tanpa trade-off penurunan EX yang signifikan |
| TEP > 0 | Ada trade-off: penurunan token disertai penurunan EX |

Tabel interpretasi itu mengasumsikan ΔT < 0. Kalau ΔT > 0 (pipeline memakai token lebih banyak dari DAIL-SQL), TEP > 0 berarti EX naik, jadi baca tanda ΔEX dan ΔT di kolomnya sebelum memakai tabel ini.

In [4]:
# Satu baris per model × perbandingan × pipeline, B = DAIL-SQL
rows = ts.tep_rows(runs, ALPHA, COMPARISONS, MODELS)
# Tampilkan tabel TEP
show(ts.tep_table(rows))

**TEP with DAIL-SQL as B**

,Model,Comparison,Pipeline,B,Status,Queries,EX DAIL-SQL,EX pipeline,ΔEX (pp),T DAIL-SQL,T pipeline,ΔT,Relative ΔT,TEP,Few-shot k,DAIL k,Source pipeline,Source DAIL
0,3B,GraphRAG K0 vs DAIL-SQL K9,1-stage,DAIL-SQL,ok,1034,66.1%,59.8%,-6.3 pp,"1,800.7",427.9,"-1,372.8",-76.2%,0.1248,0,9,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k9\hasil_dail_3B_k9...
1,3B,GraphRAG K0 vs DAIL-SQL K9,2-stage,DAIL-SQL,ok,1034,66.1%,55.0%,-11.0 pp,"1,800.7",382.3,"-1,418.4",-78.8%,0.2119,0,9,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k9\hasil_dail_3B_k9...
2,3B,GraphRAG K0 vs DAIL-SQL K0,1-stage,DAIL-SQL,ok,1034,60.6%,59.8%,-0.9 pp,448.2,427.9,-20.4,-4.5%,0.3156,0,0,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k0\hasil_dail_3B_k0...
3,3B,GraphRAG K0 vs DAIL-SQL K0,2-stage,DAIL-SQL,ok,1034,60.6%,55.0%,-5.6 pp,448.2,382.3,-65.9,-14.7%,0.6287,0,0,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k0\hasil_dail_3B_k0...
4,3B,GraphRAG K9 vs DAIL-SQL K9,1-stage,DAIL-SQL,ok,1034,66.1%,59.5%,-6.6 pp,"1,800.7",884.5,-916.1,-50.9%,0.1957,9,9,outputs\tables\result_ourmethod\k9\hasil_skrip...,outputs\tables\result_dail\k9\hasil_dail_3B_k9...
5,3B,GraphRAG K9 vs DAIL-SQL K9,2-stage,DAIL-SQL,ok,1034,66.1%,58.6%,-7.4 pp,"1,800.7",835.9,-964.8,-53.6%,0.2104,9,9,outputs\tables\result_ourmethod\k9\hasil_skrip...,outputs\tables\result_dail\k9\hasil_dail_3B_k9...
6,7B,GraphRAG K0 vs DAIL-SQL K9,1-stage,DAIL-SQL,ok,1034,75.6%,70.5%,-5.1 pp,"1,796.0",419.1,"-1,376.9",-76.7%,0.0884,0,9,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k9\hasil_dail_7B_k9...
7,7B,GraphRAG K0 vs DAIL-SQL K9,2-stage,DAIL-SQL,ok,1034,75.6%,65.9%,-9.8 pp,"1,796.0",370.2,"-1,425.8",-79.4%,0.1627,0,9,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k9\hasil_dail_7B_k9...
8,7B,GraphRAG K0 vs DAIL-SQL K0,1-stage,DAIL-SQL,ok,1034,73.1%,70.5%,-2.6 pp,438.3,419.1,-19.2,-4.4%,0.8152,0,0,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k0\hasil_dail_7B_k0...
9,7B,GraphRAG K0 vs DAIL-SQL K0,2-stage,DAIL-SQL,ok,1034,73.1%,65.9%,-7.3 pp,438.3,370.2,-68.1,-15.5%,0.6388,0,0,outputs\tables\result_ourmethod\k0\hasil_skrip...,outputs\tables\result_dail\k0\hasil_dail_7B_k0...


## 5. Simpan

`outputs/tables/hasil_tep.json` (angka mentah) dan `outputs/tables/hasil_tep.xlsx` (sheet `TEP` dan `Sources`).

In [5]:
# Simpan JSON dan Excel
json_path, xlsx_path = ts.save_results(rows, runs, notes, ALPHA, COMPARISONS, OUT_DIR)
print(f"Tersimpan: {json_path}\nTersimpan: {xlsx_path}")

Tersimpan: outputs\tables\hasil_tep.json
Tersimpan: outputs\tables\hasil_tep.xlsx
